# bottleneck-latent-projection — faded example 3: Complete the decoder un-flatten back to a feature map

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `bottleneck-latent-projection`. Running the beacon reports progress on the `Generative: Bottleneck latent projection` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Bottleneck latent projection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bottleneck-latent-projection`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bottleneck-latent-projection"
DD_SUBTOPIC = "Generative: Bottleneck latent projection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The decoder mirror expands a latent code with `Linear(latent, hidden)` + ReLU then `Linear(hidden, C*H*W)`, producing a flat reconstruction. To hand this to the transposed-conv stack it must be reshaped back to `(B, C, H, W)` with `Rearrange('b (c h w) -> b c h w', ...)`, passing the original `c/h/w` sizes so the single axis splits unambiguously.

## Faded exercise 3

Implement `decode(z, W1, b1, W2, b2, C, H, W)`. The two expand Linears and the ReLU are given, producing `flat` of shape `(B, C*H*W)`. Complete the **un-flatten** step that reshapes `flat` back into a `(B, C, H, W)` feature map using einops `rearrange` with the supplied `C`, `H`, `W` sizes.

**Fill in:** Reshapes the flat reconstruction (B, C*H*W) back to a (B, C, H, W) feature map via rearrange with the given c/h/w sizes.

In [ ]:
def decode(z, W1, b1, W2, b2, C, H, W):
    h = t.relu(z @ W1.T + b1)
    flat = h @ W2.T + b2
    x = None  # TODO: Reshape the flat reconstruction (B, C*H*W) back to a (B, C, H, W) feature map via rearrange with the given c/h/w sizes.
    return x

t.manual_seed(0)
B, C, H, W = 3, 8, 4, 4
latent, hidden = 4, 32
out_f = C * H * W
z = t.randn(B, latent)
W1 = t.randn(hidden, latent); b1 = t.randn(hidden)
W2 = t.randn(out_f, hidden); b2 = t.randn(out_f)
x = decode(z, W1, b1, W2, b2, C, H, W)
print('feature-map shape:', tuple(x.shape))


def _test():
    t.manual_seed(0)
    B, C, H, W = 3, 8, 4, 4
    latent, hidden = 4, 32
    out_f = C * H * W
    z = t.randn(B, latent)
    W1 = t.randn(hidden, latent); b1 = t.randn(hidden)
    W2 = t.randn(out_f, hidden); b2 = t.randn(out_f)
    x = decode(z, W1, b1, W2, b2, C, H, W)
    assert x.shape == (B, C, H, W), x.shape
    h = t.relu(z @ W1.T + b1)
    flat_ref = h @ W2.T + b2
    x_ref = flat_ref.reshape(B, C, H, W)
    assert t.allclose(x, x_ref, atol=1e-5), 'feature-map mismatch'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def decode(z, W1, b1, W2, b2, C, H, W):
    h = t.relu(z @ W1.T + b1)
    flat = h @ W2.T + b2
    x = rearrange(flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)
    return x

t.manual_seed(0)
B, C, H, W = 3, 8, 4, 4
latent, hidden = 4, 32
out_f = C * H * W
z = t.randn(B, latent)
W1 = t.randn(hidden, latent); b1 = t.randn(hidden)
W2 = t.randn(out_f, hidden); b2 = t.randn(out_f)
x = decode(z, W1, b1, W2, b2, C, H, W)
print('feature-map shape:', tuple(x.shape))
```
</details>